In [1]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install trl datasets accelerate peft bitsandbytes transformers sentencepiece evaluate matplotlib pandas
!pip -q install lm-eval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s 

In [2]:
import os, gc, math, time, json, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict
from transformers import TrainingArguments, EarlyStoppingCallback


from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import warnings

warnings.simplefilter('ignore')

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [6]:
def cuda_mem(label=""):
    if not torch.cuda.is_available():
        print("CUDA not available")
        return
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    max_alloc = torch.cuda.max_memory_allocated() / 1024**2
    print(f"[VRAM] {label} allocated={allocated:.0f} MB | reserved={reserved:.0f} MB | max_alloc={max_alloc:.0f} MB")

In [7]:
def reset_cuda_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

In [8]:
MODEL_NAME = "unsloth/Qwen3-0.6B"
DATASET_NAME = "Vikhrmodels/GrandMaster-PRO-MAX"

SEED = 42
MAX_SEQ_LENGTH = 2048
PACKING = True

# корзинка промптов до/после
prompts_for_test = [
    'Как вкусно приготовить индейку на гриле?',
    'Как распознать приближающийся инсульт?',
    'Сформулируй основные каноны архитектуры древних цивилизаций',
    'Облагать ли страховыми взносами суммы прощенного долга по займу от организации где работает застрахованный?',
    'Расскажи мне про Курчатова'
]

In [9]:
set_seed(SEED)
cuda_mem("start")

[VRAM] start allocated=8 MB | reserved=22 MB | max_alloc=8 MB


In [10]:
reset_cuda_peak()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

cuda_mem("after model load (4bit)")

==((====))==  Unsloth 2026.6.1: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[VRAM] after model load (4bit) allocated=596 MB | reserved=656 MB | max_alloc=612 MB


In [11]:
tokenizer.chat_template

'{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0].role == \'system\' %}\n        {{- messages[0].content + \'\\n\\n\' }}\n    {%- endif %}\n    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0].content + \'<|im_end|>\\n\' }}\n    {%- endif %}\n{%- endif %}\n{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}\n{%- for forward_message 

In [12]:
FastLanguageModel.for_inference(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024, padding_idx=151669)
    (layers): ModuleList(
      (0): Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_lay

In [13]:
def generate_chat(model, tokenizer, user_text, system_text="Ты — полезный русскоязычный ассистент.", max_new_tokens=220):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text


In [14]:
for p in prompts_for_test:
  print(p)
  print(generate_chat(model, tokenizer, p)[:1500])
  print("="*250)

Как вкусно приготовить индейку на гриле?


Both `max_new_tokens` (=220) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
Ты — полезный русскоязычный ассистент.
user
Как вкусно приготовить индейку на гриле?
assistant
<think>
Хорошо, пользователь спрашивает, как приготовить индейку на гриле. Нужно дать точные шаги, но также учесть, что индейка может быть сухой или кислорожной, и выбрать правильный метод. Сначала нужно определить, какой вид индейки использовать: сухая или кислорожная. Потом обсудить инструкцию по приготовлению. Важно упомянуть о том, что индейка готовится как в тёплой воде, но для кислорожной она лучше готовить в воде, а сухая — в молоке. Также важно указать, какие инструменты нужны, например, кухонная палочка, сковородка, тушателя. Завершить ответом, чтобы пользователь понимал, что есть варианты подготовки, и что
Как распознать приближающийся инсульт?


Both `max_new_tokens` (=220) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
Ты — полезный русскоязычный ассистент.
user
Как распознать приближающийся инсульт?
assistant
<think>
Хорошо, пользователь спрашивает, как распознать приближающийся инсульт. Нужно дать ответ, который будет содержать точные методы и признаки. Сначала вспомним, что инсульт — это острый, неподписанный катализатор, который происходит из-за недостаточности кровообращения. При приближении инсульта, человек испытывает симптомы: однократные, но сильные болевые ощущения. Важно уточнить, какие именно признаки характерны, так как нужно быть конкретным. Проверю, нет ли дополнительных деталей, которые можно уточить. Также стоит учесть, что это может быть патологичный симптом, поэтому важно дать правильный ответ без несправедливости.
</think>

При приближении ин
Сформулируй основные каноны архитектуры древних цивилизаций


Both `max_new_tokens` (=220) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
Ты — полезный русскоязычный ассистент.
user
Сформулируй основные каноны архитектуры древних цивилизаций
assistant
<think>
Хорошо, пользователь попросил сформулировать основные канны архитектуры древних цивилизаций. Нужно понять, что именно он хочет. Возможно, это связано с тем, что он изучает этнические архитектуры, и он хочет убедиться, что упоминаются ключевые элементы. Следовательно, нужно дать ответ, который будет содержать основные аспекты, такие как структура, материалы, функции, особенности, трансформация и т.д.

Проверю, какие основные категории включают в себя архитектуру древних цивилизаций: структура, материалы, функции, трансформация, стиль, облик. Убедюсь, что все эти пункты соответствуют требованиям, и думаю, что пользователь просит именно это. Теперь составлю
Облагать ли страховыми взносами суммы прощенного долга по займу от организации где работает застрахованный?


Both `max_new_tokens` (=220) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
Ты — полезный русскоязычный ассистент.
user
Облагать ли страховыми взносами суммы прощенного долга по займу от организации где работает застрахованный?
assistant
<think>
Хорошо, пользователь спрашивает, облагать ли страховыми взносы суммы прощенного долга по займу от организации, которая работает застрахованным. Нужно ответить на это.

Сначала вспомню, что застрахованные организации обычно не подлежат обязательным взносам, потому что они работают с их клиентами. Прошенные долги — это устаревшие обязательства, которые не должны быть обогреты. Зависит от того, кто является застрахующим, а именно, от организации, которая предоставляет услуги.

Надо убедиться, что ответ логично связан с правилами обязательных взносов и статей, связанных с страховой деятельностью. Также проверю, нет ли каких-то особенностей, которые я мог бы ошибаться.

После этого, нужно сделать понятный ответ, объяснив, почему да или нет,
Расскажи мне про Курчатова
system
Ты — полезный русскоязычный ассистент.
user

In [15]:
BASE_BENCH_DIR = "/content/bench_base"
os.makedirs(BASE_BENCH_DIR, exist_ok=True)

In [16]:
# TruthfulQA (быстро)
!lm_eval \
  --model hf \
  --model_args pretrained={MODEL_NAME},trust_remote_code=True \
  --tasks truthfulqa_mc2 \
  --device cuda:0 \
  --batch_size 8 \
  --limit 50 \
  --output_path {BASE_BENCH_DIR}/truthfulqa_base.json

2026-06-07:20:39:15 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-07:20:39:27 INFO     [_cli.run:388] Selected Tasks: ['truthfulqa_mc2']
2026-06-07:20:39:27 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-06-07:20:39:27 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'unsloth/Qwen3-0.6B', 'trust_remote_code': True}
2026-06-07:20:39:31 INFO     [models.huggingface:286] Using device 'cuda:0'
config.json: 100% 752/752 [00:00<00:00, 4.54MB/s]
tokenizer_config.json: 100% 10.5k/10.5k [00:00<00:00, 14.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 76.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 126MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:00<00:00, 17.4MB/s]
added_tokens.json: 100% 707/707 [00:00<00:00, 4.91MB/s]
special_tokens_map.json: 100% 614/614 [00:00

In [17]:
ds = load_dataset(DATASET_NAME)
ds

README.md:   0%|          | 0.00/9.03k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/139M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/122M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.85M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/151822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3291 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source', 'conversation', 'prompt_tokens', 'answer_tokens', 'cluster', 'prompt_lang', 'answer_lang'],
        num_rows: 151822
    })
    test: Dataset({
        features: ['source', 'conversation', 'prompt_tokens', 'answer_tokens', 'cluster', 'prompt_lang', 'answer_lang'],
        num_rows: 3291
    })
})